# 03 Baseline And Linear Models

In [1]:
# Notebook 03: Baseline, Ridge, Lasso, ElasticNet, SVR

import joblib
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.model_selection import GridSearchCV
from utils.data_utils import project_root
from utils.model_utils import regression_metrics, summarize_results

# Set project paths and output folders
ROOT = project_root()
IN_DIR = ROOT / "data" / "processed"
MODELS_DIR = ROOT / "models"
TABLE_DIR = ROOT / "reports" / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

# Loading PCA-transformed training and validation data from Notebook 02
X_train = pd.read_parquet(IN_DIR / "X_train_pca.parquet")
X_val = pd.read_parquet(IN_DIR / "X_val_pca.parquet")
y_train = pd.read_parquet(IN_DIR / "y_train.parquet")["log_ic50"]
y_val = pd.read_parquet(IN_DIR / "y_val.parquet")["log_ic50"]

# Storing validation results for all models
rows = []

# Dummy baseline
baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)
b_pred = baseline.predict(X_val)
rows.append({"model": "dummy_mean", **regression_metrics(y_val, b_pred)})
joblib.dump(baseline, MODELS_DIR / "dummy_mean.joblib")

# Ridge
ridge = GridSearchCV(Ridge(random_state=42), 
                     {"alpha": [0.01, 0.1, 1.0, 10.0, 100.0, 500.0, 1000.0]}, 
                     cv=5, scoring="neg_root_mean_squared_error")
ridge.fit(X_train, y_train)
print(f"Ridge best CV RMSE: {-ridge.best_score_:.4f}, best alpha: {ridge.best_params_}")
r_pred = ridge.best_estimator_.predict(X_val)
rows.append({"model": "ridge", "best_params": str(ridge.best_params_), 
             **regression_metrics(y_val, r_pred)})
joblib.dump(ridge.best_estimator_, MODELS_DIR / "ridge_best.joblib")

# Lasso
lasso = GridSearchCV(Lasso(random_state=42, max_iter=5000), 
                     {"alpha": [0.0001, 0.001, 0.01, 0.1, 1.0]}, 
                     cv=5, scoring="neg_root_mean_squared_error")
lasso.fit(X_train, y_train)
print(f"Lasso best CV RMSE: {-lasso.best_score_:.4f}, best alpha: {lasso.best_params_}")
l_pred = lasso.best_estimator_.predict(X_val)
rows.append({"model": "lasso", "best_params": str(lasso.best_params_), 
             **regression_metrics(y_val, l_pred)})
joblib.dump(lasso.best_estimator_, MODELS_DIR / "lasso_best.joblib")

# ElasticNet
en = GridSearchCV(
    ElasticNet(random_state=42, max_iter=10000),
    {"alpha": [0.001, 0.01, 0.1, 1.0], "l1_ratio": [0.2, 0.5, 0.8]},
    cv=5, scoring="neg_root_mean_squared_error"
)
en.fit(X_train, y_train)
print(f"ElasticNet best CV RMSE: {-en.best_score_:.4f}, best params: {en.best_params_}")
en_pred = en.best_estimator_.predict(X_val)
rows.append({"model": "elasticnet", "best_params": str(en.best_params_), 
             **regression_metrics(y_val, en_pred)})
joblib.dump(en.best_estimator_, MODELS_DIR / "elasticnet_best.joblib")

# SVR
svr = GridSearchCV(
    SVR(),
    {"C": [0.1, 1.0, 10.0, 50.0], "epsilon": [0.1, 0.5], "kernel": ["rbf"]},
    cv=5, scoring="neg_root_mean_squared_error"
)
svr.fit(X_train, y_train)
print(f"SVR best CV RMSE: {-svr.best_score_:.4f}, best params: {svr.best_params_}")
svr_pred = svr.best_estimator_.predict(X_val)
rows.append({"model": "svr", "best_params": str(svr.best_params_), 
             **regression_metrics(y_val, svr_pred)})
joblib.dump(svr.best_estimator_, MODELS_DIR / "svr_best.joblib")

# Summary
results = summarize_results(rows)
results.to_csv(TABLE_DIR / "linear_model_results.csv", index=False)
results

Ridge best CV RMSE: 1.4710, best alpha: {'alpha': 1000.0}
Lasso best CV RMSE: 1.1874, best alpha: {'alpha': 1.0}
ElasticNet best CV RMSE: 1.1769, best params: {'alpha': 1.0, 'l1_ratio': 0.8}
SVR best CV RMSE: 1.2042, best params: {'C': 1.0, 'epsilon': 0.5, 'kernel': 'rbf'}


,model,rmse,mae,r2,best_params
0,svr,1.421916,1.060536,0.127195,"{'C': 1.0, 'epsilon': 0.5, 'kernel': 'rbf'}"
1,ridge,1.438777,1.077377,0.106372,{'alpha': 1000.0}
2,elasticnet,1.439931,1.108757,0.104939,"{'alpha': 1.0, 'l1_ratio': 0.8}"
3,lasso,1.448121,1.116392,0.094728,{'alpha': 1.0}
4,dummy_mean,1.522082,1.170333,-0.000105,NaN


## Linear and Kernel Models — Results

### Models Trained
Five models were trained and compared: Dummy baseline, Ridge, Lasso, ElasticNet, and SVR.
All models used the 100 PCA components from Notebook 02 as input features.
Hyperparameters were selected using 5-fold cross-validation with RMSE as the scoring metric.

### Hyperparameter Tuning
Hyperparameters were selected using **GridSearchCV with 5-fold cross-validation**, 
scoring on negative RMSE. In each fold, the training data is split into 5 parts — 
the model trains on 4 parts and validates on the remaining 1, rotating until every 
part has been used as validation once. The hyperparameter combination with the 
lowest average RMSE across all 5 folds is selected as the best.

The following grids were searched:
- **Ridge**: alpha ∈ [0.01, 0.1, 1.0, 10.0, 100.0, 500.0, 1000.0]
- **Lasso**: alpha ∈ [0.0001, 0.001, 0.01, 0.1, 1.0]
- **ElasticNet**: alpha ∈ [0.001, 0.01, 0.1, 1.0] × l1_ratio ∈ [0.2, 0.5, 0.8]
- **SVR**: C ∈ [0.1, 1.0, 10.0, 50.0] × epsilon ∈ [0.1, 0.5] × kernel = rbf

### Evaluation Metrics
Three metrics are reported for each model:
- **RMSE** (Root Mean Squared Error): Average prediction error in log IC50 units. 
  Lower is better. Penalizes large errors more heavily than small ones.
- **MAE** (Mean Absolute Error): Average absolute prediction error in log IC50 units. 
  Lower is better. Less sensitive to outliers than RMSE. Reported as the primary 
  error metric following Tang & Gottlieb (2021), who found the error distribution 
  in drug sensitivity prediction to be non-Gaussian, making MAE more appropriate.
- **R²** (Coefficient of Determination): Proportion of variance in IC50 explained by 
  the model. Ranges from 0 to 1, higher is better. A value of 0 means the model 
  performs no better than predicting the mean (dummy baseline).

### Cross-Validation Results (on training set)
| Model | Best CV RMSE | Best Hyperparameters |
|---|---|---|
| Ridge | 1.4710 | alpha=1000.0 |
| Lasso | 1.1874 | alpha=1.0 |
| ElasticNet | 1.1769 | alpha=1.0, l1_ratio=0.8 |
| SVR | 1.2042 | C=1.0, epsilon=0.5, kernel=rbf |

ElasticNet achieved the best CV RMSE, followed closely by Lasso and SVR.
Ridge required a very high alpha (1000), indicating strong regularization was needed —
consistent with the high-dimensional PCA feature space.

### Validation Set Results
| Model | RMSE | MAE | R² |
|---|---|---|---|
| SVR | 1.422 | 1.061 | 0.127 |
| Ridge | 1.439 | 1.077 | 0.106 |
| ElasticNet | 1.440 | 1.109 | 0.105 |
| Lasso | 1.448 | 1.116 | 0.095 |
| Dummy baseline | 1.522 | 1.170 | ~0 |

SVR achieved the best validation R² (0.127), meaning it explains ~13% of the 
variance in Erlotinib IC50 from gene expression alone. All models outperformed 
the dummy baseline, confirming that gene expression carries meaningful predictive 
signal for Erlotinib sensitivity. Overall, performance is modest (R² ≈ 0.10–0.13), 
which is expected for expression-only drug sensitivity prediction.

### Key Observations

- **SVR best on validation**: SVR with RBF kernel (C=1.0) achieved the highest R² 
  on the validation set. The RBF kernel allows SVR to capture mild nonlinear 
  relationships in PCA space that purely linear models cannot. Extending the C grid 
  confirmed C=1.0 as optimal, suggesting that more aggressive margin fitting did 
  not improve generalization.

- **Regularization in Ridge and Lasso**: Ridge uses L2 regularization, which shrinks 
  all coefficients toward zero but keeps all features. Lasso uses L1 regularization, 
  which drives some coefficients to exactly zero, effectively selecting a subset of 
  features. Ridge required alpha=1000, indicating the need for strong coefficient 
  shrinkage to control variance in the PCA feature space.

- **ElasticNet l1_ratio=0.8**: ElasticNet combines both L1 and L2 penalties. An 
  l1_ratio of 0.8 means the model weighted L1 (sparsity) more heavily than L2 
  (shrinkage), suggesting the model favors sparser solutions where fewer components 
  contribute more strongly to the prediction.

- **CV vs validation gap**: ElasticNet had the best CV RMSE (1.177) but SVR won 
  on validation (R²=0.127). This small discrepancy is normal — CV selects the 
  best average performer across folds, but a single held-out validation set can 
  favor a different model. The final test set in Notebook 05 resolves this.

- **MAE reported alongside RMSE** following Tang & Gottlieb (2021), who found MAE more appropriate for IC50 prediction 
  due to the non-Gaussian (Laplacian) error distribution in drug sensitivity data.

### Saved Artifacts
- `ridge_best.joblib`, `lasso_best.joblib`, `elasticnet_best.joblib`, `svr_best.joblib`
- `linear_model_results.csv`

#### Reference:
Tang, Y.-C., & Gottlieb, A. (2021). Explainable drug sensitivity prediction 
through cancer pathway enrichment. Scientific Reports, 11, 3128. 
https://doi.org/10.1038/s41598-021-82612-7